In [1]:
import pandas as pd
import jdatetime as jdt
import jalali_pandas

In [2]:
sale = pd.read_excel(r"D:\تحلیل داده\python\تمرین\WoodInc-sale.xlsx")
purchase =pd.read_excel(r"D:\تحلیل داده\python\تمرین\WoodInc-purchase.xlsx")

In [3]:
sale.groupby('کد کالا').agg(sumsale = ('مبلغ خالص','sum')).reset_index()

,کد کالا,sumsale
0,1-1-01-004,429000018
1,1-1-01-029,830079980
2,1-1-01-034,841199984
3,1-1-01-050,345059986
4,1-1-01-053,43120000
...,...,...
732,1-1-12-026,14000000
733,1-1-12-028,97350000
734,1-1-12-029,10500001
735,1-1-12-030,10500001


In [4]:
purchase.groupby('کد کالا').agg(sumsale = ('مبلغ خالص','sum')).reset_index()

,کد کالا,sumsale
0,1-1-01-004,-378180019
1,1-1-01-029,-880868240
2,1-1-01-034,-559006500
3,1-1-01-050,-444338500
4,1-1-01-055,-359647680
...,...,...
551,1-1-12-027,-82700000
552,1-1-12-028,-85030011
553,1-1-12-029,-107510001
554,1-1-12-030,-132320001


In [5]:
def to_jalali_date(column, year_dir='L'):
    column = pd.Series(column)

    if column.dtype == 'object':
        sep = next((s for s in ['/', '-', '.'] if s in str(column.iloc[0])), None)
        mod_col = [x.split(sep) for x in column]
    
    elif pd.api.types.is_integer_dtype(column):
        str_col = column.astype(str)
        sample = str_col.iloc[0]
        if len(sample) == 8:
            if year_dir == 'L':
                mod_col = [[x[:4], x[4:6], x[6:]] for x in str_col]
            else:
                mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        elif len(sample) == 6:
            mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        else:
            raise ValueError("Unsupported integer date format")

    else:
        raise TypeError("Unsupported column dtype")

    # Extract year, month, day based on year direction
    idx = (0, 1, 2) if year_dir == 'L' else (2, 1, 0)
    year, month, day = zip(*[(i[idx[0]], i[idx[1]], i[idx[2]]) for i in mod_col])

    # Fix 2-digit years
    year = [
        f"14{y}" if len(y) == 2 and int(y) < 50 else
        f"13{y}" if len(y) == 2 and int(y) >= 50 else
        y
        for y in year
    ]

    return [jdt.date(int(y), int(m), int(d)) for y, m, d in zip(year, month, day)]

In [6]:
sale['shamsi'] = to_jalali_date(sale['تاريخ'])
sale['shamsi_yearsale'] =sale.shamsi.jalali.year
sale['shamsi_monthsale'] =sale.shamsi.jalali.month
purchase['shamsi'] = to_jalali_date(purchase['تاريخ'])
purchase['shamsi_yearpurchase'] =purchase.shamsi.jalali.year
purchase['shamsi_monthpurchase'] =purchase.shamsi.jalali.month

In [7]:
totalsale =sale.groupby(['کد کالا','shamsi_yearsale']).agg(sumquantity_sale=('مقدار','sum')).reset_index()
totalpurchase = purchase.groupby(['کد کالا','shamsi_yearpurchase']).agg(sumquantity_purchase=('مقدار','sum')).reset_index()
total=pd.merge(left=totalpurchase,right=totalsale, on='کد کالا')
total['remaining_stock']=(total['sumquantity_purchase']-total['sumquantity_sale'])
total

,کد کالا,shamsi_yearpurchase,sumquantity_purchase,shamsi_yearsale,sumquantity_sale,remaining_stock
0,1-1-01-004,1399,66.0,1399,66.0,0.0
1,1-1-01-029,1399,68.0,1399,125.0,-57.0
2,1-1-01-034,1399,39.0,1399,86.0,-47.0
3,1-1-01-050,1399,31.0,1399,35.0,-4.0
4,1-1-01-055,1399,30.0,1399,81.0,-51.0
...,...,...,...,...,...,...
517,1-1-12-026,1399,10.0,1399,2.0,8.0
518,1-1-12-028,1399,11.0,1399,11.0,0.0
519,1-1-12-029,1399,13.0,1399,1.0,12.0
520,1-1-12-030,1399,16.0,1399,1.0,15.0


In [8]:
totalsale =sale.groupby(['کد کالا','shamsi_monthsale']).agg(sumquantity_sale=('مقدار','sum')).reset_index()
totalpurchase = purchase.groupby(['کد کالا','shamsi_monthpurchase']).agg(sumquantity_purchase=('مقدار','sum')).reset_index()
total1=pd.merge(left=totalpurchase,right=totalsale, on='کد کالا')
total1['remaining_stock']=(total1['sumquantity_purchase']-total1['sumquantity_sale'])
total1

,کد کالا,shamsi_monthpurchase,sumquantity_purchase,shamsi_monthsale,sumquantity_sale,remaining_stock
0,1-1-01-004,4,66.0,4,66.0,0.0
1,1-1-01-029,7,24.0,1,6.0,18.0
2,1-1-01-029,7,24.0,2,12.0,12.0
3,1-1-01-029,7,24.0,3,27.0,-3.0
4,1-1-01-029,7,24.0,4,43.0,-19.0
...,...,...,...,...,...,...
8310,1-1-12-031,7,34.0,7,13.0,21.0
8311,1-1-12-031,7,34.0,8,3.0,31.0
8312,1-1-12-031,7,34.0,9,3.0,31.0
8313,1-1-12-031,7,34.0,10,4.0,30.0
